# سامانهٔ تشخیص پوششِ صورت — نسخهٔ پایه (منظم‌شده)

این **دقیقاً همان الگوریتمِ نسخهٔ اول** است. هیچ آستانه، ضریب یا شرطی
عوض نشده. فقط به بخش‌های مشخص تقسیم شده تا هر تکه معلوم باشد چه
کاری می‌کند — پایه‌ای برای اینکه قدم‌به‌قدم رویش قابلیت اضافه کنی.

## خط لولهٔ کلی

```
ویدیو
  └─ ۱) YOLO11-pose ────────► جعبهٔ فرد + ۱۷ کی‌پوینت + شناسهٔ ردیابی
      └─ گیت: میانگین اطمینانِ (بینی، چشم چپ، چشم راست) ≥ ۰.۵ ؟
          │      نه → این فرد در این فریم کاملاً نادیده گرفته می‌شود
          └─ بله
              └─ ۴) تراز روی خطِ چشم + برشِ صورت
                  └─ ۲) طبقه‌بندِ ماسک ──► mask / no_mask
                      └─ اگر ماسک داشت:
                          ۳) نسبتِ پوست در ۴۵٪ بالای برش < ۰.۱۲ ؟
                              بله → قرمز (مشکوک)   نه → نارنجی (پزشکی)
                          └─ ۵) رأی‌گیری زمانی → رنگِ نهایی
```

## جدول تصمیم

| رنگ | برچسب | معنی |
|---|---|---|
| ⚪ خاکستری | `Analyzing...` | هنوز رأی کافی جمع نشده |
| 🟢 سبز | `Clear` | صورت باز — **قفلِ دائمی**، دیگر بررسی نمی‌شود |
| 🟠 نارنجی | `Medical Mask` | ماسک دارد ولی پوستِ بالای صورت پیداست |
| 🔴 قرمز | `SUSPICIOUS - ALERT` | ماسک دارد و بالای صورت هم پوشیده است |

## ساختار نوت‌بوک

| بخش | کار |
|---|---|
| ۰ | نصب و راه‌اندازی |
| ۱ | مدل ژست (تشخیص فرد + اسکلت + ردیابی) |
| ۲ | مدل ماسک |
| ۳ | سنجهٔ پوست — تفکیک پزشکی از مشکوک |
| ۴ | تراز و برشِ صورت |
| ۵ | ماشینِ حالت — رأی‌گیری زمانی |
| ۶ | ابزار نمایش — گالری و اسکلت |
| ۷ | خط لولهٔ اصلی |
| ۸ | اجرا |

> **قبل از شروع:** Runtime ← Change runtime type ← GPU

---
## ۰) نصب و راه‌اندازی

In [ ]:
!pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow

### وارد کردن کتابخانه‌ها و انتخاب دستگاه

`USE_HALF` یعنی محاسبات با دقتِ نصف (FP16). روی GPU تقریباً دو برابر
سریع‌تر است و برای این کار افتِ دقتِ محسوسی ندارد.

In [ ]:
import cv2, time, torch, numpy as np
from collections import deque
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, SiglipForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_HALF = device == "cuda"
print(f"🔧 Device: {device} | FP16: {USE_HALF}")

---
## ۱) مدل ژست — تشخیص فرد، اسکلت و ردیابی

یک مدل، سه کار همزمان:

- **جعبهٔ هر فرد** در تصویر
- **۱۷ کی‌پوینت** استاندارد COCO برای هر فرد
- **شناسهٔ پایدار** برای دنبال‌کردن هر نفر بین فریم‌ها (ByteTrack)

اینکه هر سه از یک مدل بیرون می‌آید مهم است: اگر تشخیصِ فرد و تخمینِ
ژست دو مدلِ جدا بودند، تصویر باید دو بار encode می‌شد.

**اندیس‌های کی‌پوینت** پایین تعریف شده‌اند تا هیچ‌جای کد عددِ جادویی
ننویسیم. `UPPER_BODY_SKELETON` فقط بالاتنه است — سر، شانه، آرنج و مچ.
پایین‌تنه برای تشخیصِ سرقت بی‌فایده است و فقط شلوغی بصری می‌آورد.

In [ ]:
# ---------------- 1) Single Unified Model: Person+Pose+Track ----------
pose_model = YOLO("yolo11n-pose.pt")

NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4
LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST = 5, 6, 7, 8, 9, 10

# اسکلت فقط بالاتنه (سر، شونه، بازو، ساعد) - برای جلوه بصری
UPPER_BODY_SKELETON = [
    (LEYE, REYE), (NOSE, LEYE), (NOSE, REYE),
    (LEAR, LEYE), (REAR, REYE),
    (LSHOULDER, RSHOULDER),
    (LSHOULDER, LELBOW), (LELBOW, LWRIST),
    (RSHOULDER, RELBOW), (RELBOW, RWRIST),
    (NOSE, LSHOULDER), (NOSE, RSHOULDER),
]

---
## ۲) طبقه‌بندِ ماسک

یک مدلِ **دوکلاسهٔ فاین‌تیون‌شده** روی همین تسک: `mask` یا `no_mask`.
چون روی همین مسئله آموزش دیده، روی برش‌های کوچکِ دوربین مداربسته
قابل‌اتکاست.

`classify_mask_batch` عمداً **دسته‌ای** کار می‌کند: همهٔ صورت‌های یک
فریم با هم به مدل داده می‌شوند، نه یکی‌یکی. روی GPU این تفاوتِ بزرگی
در سرعت می‌سازد.

> **نکته برای مرحلهٔ بعد:** `MASK_ID2LABEL` دستی نوشته شده. اگر ترتیبِ
> کلاس‌ها در چک‌پوینت برعکس باشد، کلِ سیستم وارونه کار می‌کند **بدون
> اینکه هیچ خطایی بدهد**. برای اطمینان یک بار این را اجرا کن:
> `print(mask_model.config.id2label)`

In [ ]:
# ---------------- 2) Mask Classifier (Pretrained, FP16) ----------------
MASK_MODEL_NAME = "prithivMLmods/Face-Mask-Detection"
mask_processor = AutoImageProcessor.from_pretrained(MASK_MODEL_NAME)
mask_model = SiglipForImageClassification.from_pretrained(MASK_MODEL_NAME).to(device).eval()
if USE_HALF:
    mask_model = mask_model.half()
MASK_ID2LABEL = {0: "mask", 1: "no_mask"}

@torch.no_grad()
def classify_mask_batch(face_list):
    if len(face_list) == 0:
        return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in face_list]
    inputs = mask_processor(images=pil_imgs, return_tensors="pt").to(device)
    if USE_HALF:
        inputs = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in inputs.items()}
    logits = mask_model(**inputs).logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    out = []
    for p in probs:
        pred = int(np.argmax(p))
        out.append((MASK_ID2LABEL[pred], float(p[pred])))
    return out

---
## ۳) سنجهٔ پوست — تفکیکِ ماسکِ پزشکی از مشکوک

مدلِ بخش ۲ فقط می‌گوید «ماسک دارد یا نه». ولی برای امنیت، ماسکِ پزشکی
و ماسکِ دزدی دو چیزِ کاملاً متفاوت‌اند.

منطقِ این بخش: اگر کسی ماسکِ پزشکی زده، **بالای صورتش پوست دیده
می‌شود** (پیشانی، دور چشم). اگر پوششِ کامل داشته باشد، آنجا هم
پوشیده است.

- `skin_ratio` — چه نسبتی از پیکسل‌های یک ناحیه در محدودهٔ رنگیِ پوست
  قرار دارند (فضای HSV)
- `is_suspicious` — همین نسبت را روی **۴۵٪ بالای برش** حساب می‌کند؛
  اگر زیر ۰.۱۲ بود یعنی بالای صورت هم پوشیده است → مشکوک

In [ ]:
# ---------------- 3) Skin-ratio heuristic (Medical vs Suspicious) -----
def skin_ratio(region_bgr):
    if region_bgr is None or region_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 40], dtype=np.uint8)
    upper = np.array([25, 180, 255], dtype=np.uint8)
    m = cv2.inRange(hsv, lower, upper)
    return float(np.count_nonzero(m)) / m.size

def is_suspicious(face_bgr):
    h, w, _ = face_bgr.shape
    upper = face_bgr[0:int(h * 0.45), :]
    return skin_ratio(upper) < 0.12

---
## ۴) تراز و برشِ صورت

اینجا دو کار انجام می‌شود:

**الف) گیتِ ورودی.** میانگینِ اطمینانِ سه کی‌پوینتِ بینی و دو چشم حساب
می‌شود. اگر زیر `conf_th` بود، تابع `None` برمی‌گرداند و آن فرد در آن
فریم **هیچ رأیی ثبت نمی‌کند**. این همان چیزی است که باعث می‌شود
نیم‌رخ‌ها و افرادِ پشت‌به‌دوربین اصلاً قضاوت نشوند.

**ب) تراز و برش.** تصویر حولِ مرکزِ چشم‌ها می‌چرخد تا خطِ چشم افقی شود،
بعد کادری به عرضِ `۳.۲ × فاصلهٔ دو چشم` و ارتفاعِ `۳.۹ ×` آن برداشته
می‌شود — از ۱.۳ برابر بالای خطِ چشم تا ۲.۶ برابر پایینِ آن، که بینی و
دهان و چانه را می‌پوشاند.

> **نکته برای مرحلهٔ بعد:** زاویهٔ چرخش از
> `atan2(reye − leye)` گرفته می‌شود. در استاندارد COCO، `LEYE` چشمِ
> چپِ **خودِ شخص** است که در تصویر سمتِ راست دیده می‌شود — پس برای
> صورتِ روبه‌رو این اختلاف منفی می‌شود و زاویه نزدیکِ ۱۸۰ درجه
> درمی‌آید. ارزش دارد یک بار برش‌ها را ذخیره کنی و با چشم ببینی.

In [ ]:
# ---------------- 4) Keypoint-based Face Visibility + Align + Crop ----
def align_and_crop_face(person_img, kxy, kconf, conf_th=0.5):
    nose_c, leye_c, reye_c = kconf[NOSE], kconf[LEYE], kconf[REYE]
    face_score = float((nose_c + leye_c + reye_c) / 3.0)
    if face_score < conf_th:
        return None, face_score

    leye, reye = kxy[LEYE], kxy[REYE]
    eye_dist = float(np.linalg.norm(np.array(leye) - np.array(reye)))
    if eye_dist < 3:
        return None, face_score

    h, w = person_img.shape[:2]
    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eye_center = ((leye[0] + reye[0]) / 2.0, (leye[1] + reye[1]) / 2.0)

    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    rotated = cv2.warpAffine(person_img, M, (w, h))

    half_w = eye_dist * 1.6
    top = eye_center[1] - eye_dist * 1.3
    bottom = eye_center[1] + eye_dist * 2.6

    x1, x2 = int(max(0, eye_center[0] - half_w)), int(min(w, eye_center[0] + half_w))
    y1, y2 = int(max(0, top)), int(min(h, bottom))
    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return None, face_score

    return rotated[y1:y2, x1:x2], face_score

---
## ۵) ماشینِ حالت — رأی‌گیریِ زمانی

هستهٔ تصمیم. به‌جای اینکه هر فریم مستقل حکم بدهد (که باعثِ چشمک‌زدنِ
رنگ‌ها می‌شود)، رأی‌ها روی چند فریم جمع می‌شوند.

**دو حالت:**

| حالت | نرخِ بررسی | رفتار |
|---|---|---|
| `fast` | هر فریم | فردِ تازه‌وارد — تا ۳ رأی جمع شود |
| `focus` | هر ۲ فریم | فردِ ماسک‌دار/مشکوک — زیرِ نظر می‌ماند |

**در حالت `fast`:** سه رأی جمع می‌شود و امتیازِ هر رنگ برابرِ **مجموعِ
اطمینانِ** رأی‌های آن رنگ است (نه صرفاً شمارش). رنگِ برنده:

- سبز → `_lock` → **قفلِ دائمی**، از این به بعد اصلاً پردازش نمی‌شود
- نارنجی یا قرمز → می‌رود به حالت `focus`

**در حالت `focus`:** پنجرهٔ چرخشیِ ۴تایی. اگر ۳ رأی سبز در پنجره جمع
شود، به سبز قفل می‌شود؛ وگرنه رنگِ غالب بین نارنجی و قرمز نمایش
داده می‌شود.

`is_new` و `just_finalized` هیچ نقشی در تصمیم ندارند — فقط برای
فعال‌کردنِ جلوه‌های نمایشیِ بخش ۶ هستند.

> **نکته برای مرحلهٔ بعد:** قفلِ سبز **دائمی** است. یعنی کسی که با
> صورتِ باز وارد شود و بعد ماسک بکشد، تا آخرِ ویدیو سبز می‌ماند.

In [ ]:
# ---------------- 5) Smart State Manager (same logic as v4) -----------
COLORS = {"gray": (160, 160, 160), "green": (0, 200, 0),
          "orange": (0, 140, 255), "red": (0, 0, 255)}
LABELS = {"gray": "Analyzing...", "green": "Clear",
          "orange": "Medical Mask", "red": "SUSPICIOUS - ALERT"}

class TrackStateManager:
    def __init__(self):
        self.data = {}
        self.FAST_VOTES_NEEDED = 3
        self.FOCUS_INTERVAL = 2
        self.FOCUS_WINDOW = 4
        self.FOCUS_GREEN_NEEDED = 3

    def ensure(self, tid):
        if tid not in self.data:
            self.data[tid] = {
                "mode": "fast", "locked": False, "votes": [],
                "focus_window": deque(maxlen=self.FOCUS_WINDOW),
                "color": "gray", "label": LABELS["gray"],
                "is_new": True,        # برای افکت نمایشی: آیا تازه معرفی شده؟
                "just_finalized": None # برای افکت نمایشی: آیا همین الان قفل نهایی گرفته؟
            }
        return self.data[tid]

    def should_analyze(self, tid, frame_idx):
        st = self.data[tid]
        if st["locked"]:
            return False
        if st["mode"] == "fast":
            return True
        return frame_idx % self.FOCUS_INTERVAL == 0

    def register_vote(self, tid, category, conf):
        st = self.data[tid]
        st["just_finalized"] = None

        if st["mode"] == "fast":
            st["votes"].append((category, conf))
            if len(st["votes"]) >= self.FAST_VOTES_NEEDED:
                score = {"green": 0.0, "orange": 0.0, "red": 0.0}
                for c, cf in st["votes"]:
                    score[c] += cf
                best = max(score, key=score.get)
                if best == "green":
                    self._lock(tid, "green")
                else:
                    st["mode"] = "focus"
                    st["color"], st["label"] = best, LABELS[best]
                    st["focus_window"].append(best)
                    if best == "red":
                        st["just_finalized"] = "red"
        else:
            st["focus_window"].append(category)
            window = list(st["focus_window"])
            green_count = window.count("green")
            if green_count >= self.FOCUS_GREEN_NEEDED:
                self._lock(tid, "green")
            else:
                sub = [c for c in window if c in ("orange", "red")]
                if sub:
                    best = max(set(sub), key=sub.count)
                    if best == "red" and st["color"] != "red":
                        st["just_finalized"] = "red"
                    st["color"], st["label"] = best, LABELS[best]

    def _lock(self, tid, category):
        st = self.data[tid]
        st["locked"] = True
        st["color"] = category
        st["label"] = LABELS[category]

state_mgr = TrackStateManager()

---
## ۶) ابزارِ نمایش — گالری و اسکلت

این بخش روی تصمیم‌گیری هیچ اثری ندارد؛ فقط خروجی را قابلِ ارائه
می‌کند.

- **`PresentationGallery`** — چند تصویرِ کوچک در گوشهٔ تصویر: آخرین
  افرادِ شناسایی‌شده. در ارائه به کارفرما بیشترین اثر را دارد، چون
  نشان می‌دهد سیستم فقط هشدار نمی‌دهد بلکه **چهرهٔ سوژه را بیرون
  می‌کشد**.
- **`draw_upper_skeleton`** — اسکلتِ بالاتنه روی تصویر.

In [ ]:
# ---------------- 6) Presentation Helpers (Gallery + Skeleton) --------
class PresentationGallery:
    """گالری تصاویر کوچک در گوشه تصویر - آخرین افراد شناسایی‌شده"""
    def __init__(self, max_items=4, thumb_size=140):
        self.items = deque(maxlen=max_items)   # هر آیتم: (img, label, color)
        self.thumb_size = thumb_size

    def add(self, crop_bgr, label, color):
        if crop_bgr is None or crop_bgr.size == 0:
            return
        thumb = cv2.resize(crop_bgr, (self.thumb_size, self.thumb_size))
        self.items.append((thumb, label, color))

    def draw(self, frame):
        h, w = frame.shape[:2]
        pad = 10
        for i, (thumb, label, color) in enumerate(self.items):
            x2 = w - pad
            x1 = x2 - self.thumb_size
            y1 = pad + i * (self.thumb_size + 35)
            y2 = y1 + self.thumb_size
            if y2 > h:
                break
            frame[y1:y2, x1:x2] = thumb
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            cv2.putText(frame, label, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

gallery = PresentationGallery(max_items=4, thumb_size=140)

def draw_upper_skeleton(frame, kxy, kconf, color=(255, 255, 0), conf_th=0.4):
    """رسم اسکلت بالاتنه فقط - برای جلوه بصری خفن"""
    for a, b in UPPER_BODY_SKELETON:
        if kconf[a] < conf_th or kconf[b] < conf_th:
            continue
        pa = tuple(map(int, kxy[a]))
        pb = tuple(map(int, kxy[b]))
        cv2.line(frame, pa, pb, color, 2)
    for idx in [NOSE, LEYE, REYE, LEAR, REAR, LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST]:
        if kconf[idx] >= conf_th:
            p = tuple(map(int, kxy[idx]))
            cv2.circle(frame, p, 4, color, -1)

---
## ۷) خط لولهٔ اصلی

نسخهٔ اصلی دو تابعِ تقریباً یکسان داشت: `process_video` و
`process_video_demo`. بررسی کردم — منطقِ تصمیم در هر دو **دقیقاً یکی**
بود و نسخهٔ دمو فقط سه چیزِ نمایشی اضافه داشت. پس اینجا یکی شده‌اند و
آن سه چیز با پرچم کنترل می‌شوند:

| پرچم | پیش‌فرض | اثر |
|---|---|---|
| `show_skeleton` | `True` | رسمِ اسکلتِ بالاتنه |
| `show_gallery` | `True` | گالریِ گوشهٔ تصویر |
| `slowmo_repeat` | `6` | تکرارِ فریم در لحظاتِ کلیدی (`1` = خاموش) |

با `show_skeleton=False, show_gallery=False, slowmo_repeat=1` دقیقاً
همان رفتارِ `process_video`ِ ساده را می‌گیری.

**ترتیبِ کار در هر فریم:**

1. گرفتنِ جعبه‌ها، کی‌پوینت‌ها و شناسه‌ها از مدلِ ژست
2. برای هر فرد: اگر ماشینِ حالت اجازه داد، برشِ صورت ساخته می‌شود
3. همهٔ برش‌های این فریم **یکجا** به طبقه‌بند می‌روند
4. رأی هر نفر ثبت می‌شود
5. کادرها و برچسب‌ها کشیده می‌شوند
6. فریم در فایلِ خروجی نوشته می‌شود

سه بررسیِ ایمنی هم اضافه شده که **هیچ اثری روی الگوریتم ندارند** و فقط
جلوی شکستِ بی‌صدا را می‌گیرند — با `# [ایمنی]` علامت‌گذاری شده‌اند.

In [ ]:
# ---------------- 7) Main Pipeline ------------------------------------
def process_video(input_path, output_path, conf_thres=0.4, yolo_imgsz=640,
                  face_conf_th=0.5, slowmo_repeat=6,
                  show_skeleton=True, show_gallery=True):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    if not out.isOpened():                                        # [ایمنی]
        raise RuntimeError(f"❌ فایل خروجی باز نشد: {output_path}\n"
                           "   مسیر را بررسی کن (روی ویندوز /content/ معتبر نیست).")
    t0 = time.time()
    last_pct = -1
    frame_idx = 0
    seen_ids = set()   # برای تشخیص "فرد کاملا جدید" جهت افکت اسلوموشن

    results_gen = pose_model.track(
        source=input_path, classes=[0], conf=conf_thres, imgsz=yolo_imgsz,
        tracker="bytetrack.yaml", stream=True, verbose=False, persist=True,
        half=USE_HALF
    )

    for r in results_gen:
        frame_idx += 1
        frame = r.orig_img

        if r.boxes.id is None or r.keypoints is None:
            out.write(frame)
            pct = int(frame_idx / total * 100) if total > 0 else 0
            if pct != last_pct and pct % 5 == 0:
                print(f"⏳ Progress: {pct}%"); last_pct = pct
            continue

        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kpts_xy_all = r.keypoints.xy.cpu().numpy()
        kpts_conf_all = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(kpts_xy_all.shape[:2])

        batch_crops, batch_meta = [], []
        trigger_slowmo = False   # آیا این فریم باید کند نمایش داده بشه؟

        for tid, box, kxy, kconf in zip(ids, boxes, kpts_xy_all, kpts_conf_all):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            st = state_mgr.ensure(tid)

            # --- افکت نمایشی: فرد کاملا جدید -> اسلوموشن + اضافه به گالری ---
            if tid not in seen_ids:
                seen_ids.add(tid)
                trigger_slowmo = True
                square_crop = person_crop.copy()
                gallery.add(square_crop, f"New Person ID {tid}", (255, 255, 0))

            if state_mgr.should_analyze(tid, frame_idx):
                local_kxy = kxy.copy()
                local_kxy[:, 0] -= x1
                local_kxy[:, 1] -= y1

                face_crop, face_score = align_and_crop_face(person_crop, local_kxy, kconf, face_conf_th)

                if face_crop is not None:
                    batch_crops.append(face_crop)
                    batch_meta.append(tid)

            # --- رسم اسکلت بالاتنه (جلوه بصری) ---
            if show_skeleton:
                local_kxy_full = kxy.copy()
                draw_upper_skeleton(frame, local_kxy_full, kconf)

        if batch_crops:
            results = classify_mask_batch(batch_crops)
            for (tid, (mask_label, conf), face_bgr) in zip(batch_meta, results, batch_crops):
                if mask_label == "no_mask":
                    state_mgr.register_vote(tid, "green", conf)
                else:
                    cat = "red" if is_suspicious(face_bgr) else "orange"
                    state_mgr.register_vote(tid, cat, conf)

        # ---- رسم باکس‌ها + تشخیص لحظه هشدار قرمز برای اسلوموشن و گالری ----
        for tid, box in zip(ids, boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            st = state_mgr.ensure(tid)
            color = COLORS[st["color"]]
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {tid}: {st['label']}", (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if st["color"] == "red":
                cv2.putText(frame, "ALERT!", (x1, y2 + 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            if st.get("just_finalized") == "red":
                trigger_slowmo = True
                x1c, y1c = max(0, x1), max(0, y1)
                x2c, y2c = min(W, x2), min(H, y2)
                crop = frame[y1c:y2c, x1c:x2c].copy()
                gallery.add(crop, f"SUSPECT ID {tid}", (0, 0, 255))
                st["just_finalized"] = None  # فقط یکبار افکت رو نشون بده

        # ---- گالری گوشه تصویر ----
        if show_gallery:
            gallery.draw(frame)

        # ---- نوشتن خروجی (با افکت اسلوموشن در لحظات کلیدی) ----
        repeat = slowmo_repeat if trigger_slowmo else 1
        for _ in range(repeat):
            out.write(frame)

        pct = int(frame_idx / total * 100) if total > 0 else 0
        if pct != last_pct and pct % 5 == 0:
            print(f"⏳ Progress: {pct}%")
            last_pct = pct

    out.release()
    print(f"✅ Done in {time.time() - t0:.2f}s -> {output_path}")

---
## ۸) اجرا

### اتصال Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### تنظیم مسیرها و اجرا

> **نکته دربارهٔ مسیرِ ویندوز:** در نسخهٔ اصلی مسیر به‌صورت
> `"C:\1\1_پروژه\..."` نوشته شده بود. در پایتون `\1` یک کاراکترِ
> کنترلی است، نه بک‌اسلش+یک — آن مسیر هیچ‌وقت باز نمی‌شد. همیشه از
> اسلشِ رو به جلو (`/`) استفاده کن.

In [ ]:
# ---------------- مسیرها ----------------
INPUT_VIDEO  = "/content/drive/MyDrive/9.mp4"     # ← ویدیوی خودت
OUTPUT_VIDEO = "/content/output_demo.mp4"

# ---------------- پارامترها (همان مقادیرِ نسخهٔ اصلی) ----------------
CONF_THRES    = 0.4     # حداقل اطمینان برای تشخیص فرد
YOLO_IMGSZ    = 640     # اندازهٔ ورودی مدل ژست
FACE_CONF_TH  = 0.5     # گیتِ صورت: میانگین اطمینانِ بینی و دو چشم

# ---------------- جلوه‌های نمایشی ----------------
SHOW_SKELETON = True
SHOW_GALLERY  = True
SLOWMO_REPEAT = 6       # ۱ = خاموش

import os
if not os.path.exists(INPUT_VIDEO):                               # [ایمنی]
    raise FileNotFoundError(f"❌ ویدیو پیدا نشد: {INPUT_VIDEO}")

# ★ حالتِ داخلی را قبل از هر اجرا صفر می‌کنیم.
#   بدون این، اگر سلول را دو بار اجرا کنی شناسه‌ها و رأی‌های اجرای
#   قبلی باقی می‌مانند و نتیجه اشتباه می‌شود.
state_mgr = TrackStateManager()
gallery   = PresentationGallery(max_items=4, thumb_size=140)

process_video(INPUT_VIDEO, OUTPUT_VIDEO,
              conf_thres=CONF_THRES,
              yolo_imgsz=YOLO_IMGSZ,
              face_conf_th=FACE_CONF_TH,
              slowmo_repeat=SLOWMO_REPEAT,
              show_skeleton=SHOW_SKELETON,
              show_gallery=SHOW_GALLERY)

### نمایش ویدیوی خروجی

In [ ]:
import subprocess, os
from base64 import b64encode
from IPython.display import HTML

# مرورگر کدکِ mp4v را پخش نمی‌کند — با H.264 بازکدگذاری می‌کنیم
web = "/content/preview.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUTPUT_VIDEO,
                "-vcodec", "libx264", "-crf", "26", web], check=False)

path = web if os.path.exists(web) else OUTPUT_VIDEO
data = b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=860 controls>'
     f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

### ذخیره در Google Drive

In [ ]:
import shutil, os

DRIVE_DEST = '/content/drive/MyDrive/output_demo_saved.mp4'

if os.path.exists(OUTPUT_VIDEO):
    shutil.copy(OUTPUT_VIDEO, DRIVE_DEST)
    print(f"✅ ذخیره شد: {DRIVE_DEST}")
else:
    print("❌ فایل خروجی پیدا نشد — اول سلولِ اجرا را ران کن.")

---
## قدم‌های بعدی — به ترتیبِ اثر

حالا که هر بخش جداست، می‌توانی یکی‌یکی اضافه کنی. پیشنهادم به این
ترتیب، از کم‌ریسک به پرریسک:

### گام ۱ — اول ببین مدل چه می‌بیند (بدون تغییرِ الگوریتم)

قبل از هر تغییری، برش‌هایی که به طبقه‌بند می‌روند را ذخیره کن و با
چشم نگاهشان کن. در `process_video`، بعد از ساختنِ `face_crop`:

```python
cv2.imwrite(f"/content/crops/{frame_idx:05d}_{tid}.jpg", face_crop)
```

تقریباً همیشه مشکل همان‌جا دیده می‌شود. این تنها کاری است که **قبل**
از بقیه باید انجام شود.

### گام ۲ — راستی‌آزماییِ نگاشتِ لیبل (یک خط)

```python
print(mask_model.config.id2label)
```

اگر ترتیب با `MASK_ID2LABEL` نمی‌خواند، کلِ سیستم وارونه است.

### گام ۳ — برش از فریمِ تمیز

الان برش‌های گالری از فریمی گرفته می‌شوند که مستطیل و اسکلت رویش
کشیده شده. یک `clean = frame.copy()` در ابتدای حلقه و گرفتنِ همهٔ
برش‌ها از آن، تصاویرِ ارائه را تمیز می‌کند.

### گام ۴ — جهتِ چرخش در `align_and_crop_face`

بر اساس آنچه در گام ۱ دیدی. اگر برش‌ها وارونه بودند، مرتب‌سازی بر
اساس مختصاتِ x در تصویر مشکل را حل می‌کند.

### گام ۵ — شکستنِ قفلِ سبز

یک بازبینیِ کم‌هزینه هر چند ثانیه، به‌جای قفلِ دائمی.

### گام ۶ — حداقلِ اندازهٔ صورت

الان تنها گارد `eye_dist < 3` است. صورتِ ۱۵ پیکسلی هم رأی می‌دهد و
آن رأی نویزِ خالص است.

---

**یک توصیه:** بعد از هر گام، روی همان ویدیو اجرا کن و نتیجه را با
قبلش مقایسه کن. اگر چند تغییر را با هم اعمال کنی و نتیجه بدتر شود،
نمی‌فهمی کدامش مقصر بوده — همان اتفاقی که در بازنویسیِ قبلی افتاد.